In [1]:
import sys
print(sys.executable)

/usr/local/python38/bin/python


In [2]:
import llava

[2026-05-14 17:32:35,467] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


2026-05-14 17:32:36.898010: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-14 17:32:36.965019: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-14 17:32:39.184822: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
import ipywidgets
ipywidgets.__version__

'8.1.8'

In [4]:
# code in this notebook is partialy from Simon's and Nicolai's paper
# code in this notebook is partialy from llava library github

In [5]:
# CUDA_VISIBLE_DEVICES=""

In [6]:
import json
with open('./data/meetup-turns.json', 'r') as f:
    meetup_turns = json.load(f)

In [7]:
len(meetup_turns)

430

In [8]:
from llava.model.builder import load_pretrained_model
from llava.mm_utils import (
    process_images,
    tokenizer_image_token,
    get_model_name_from_path,
)

import torch


#import os
#os.environ['CUDA_VISIBLE_DEVICES'] = '0, 1'

In [9]:
#model_path = "liuhaotian/llava-v1.6-34b"
model_path = "liuhaotian/llava-v1.5-7b"

device = 'cuda:1'

from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
    IMAGE_PLACEHOLDER,
)

vis_context_path = './data/inputs/'

In [10]:
from llava.eval.run_llava import eval_model
from PIL import Image

In [11]:
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name=get_model_name_from_path(model_path),
    device=device,
    load_4bit=True,
    #load_8bit=True
)

/home/gusloryst@GU.GU.SE/.local/lib/python3.8/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/gusloryst@GU.GU.SE/.local/lib/python3.8/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [12]:
from llava.conversation import conv_templates
from matplotlib import pyplot as plt

In [13]:
# example = Image.open("../../cats_in_love.jpg")
# plt.imshow(example)
# plt.axis("off")
# plt.show()
# image_tensor = process_images([example], image_processor, model.config)
# image_tensor = image_tensor.to(model.device, dtype=torch.float16)
# prompt = "Describe this image."
# conv = conv_templates["llava_v1"].copy()
# conv.append_message(conv.roles[0], prompt)
# conv.append_message(conv.roles[1], None)

# input_ids = tokenizer_image_token(
#     conv.get_prompt(),
#     tokenizer,
#     IMAGE_TOKEN_INDEX,
#     return_tensors="pt"
# ).unsqueeze(0).to(model.device)

# output_ids = model.generate(
#     input_ids,
#     images=image_tensor,
#     max_new_tokens=200
# )

# print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

In [14]:
# import torch
# from PIL import Image

# from llava.model.builder import load_pretrained_model
# from llava.mm_utils import process_images, tokenizer_image_token
# from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN

# model_path = "liuhaotian/llava-v1.5-7b"

# tokenizer, model, image_processor, context_len = load_pretrained_model(
#     model_path, None, model_name="llava_v1_5"
# )

# image = Image.open("../../example.jpg").convert("RGB")

# image_tensor = process_images([image], image_processor, model.config)
# image_tensor = image_tensor.to(model.device, dtype=torch.float16)

# prompt = DEFAULT_IMAGE_TOKEN + "\nHow many pictures are there?"

# input_ids = tokenizer_image_token(
#     prompt,
#     tokenizer,
#     IMAGE_TOKEN_INDEX,
#     return_tensors="pt"
# ).unsqueeze(0).to(model.device)

# output_ids = model.generate(
#     input_ids,
#     images=image_tensor,
#     max_new_tokens=200
# )

# print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

In [15]:
import re
import evaluate

bleu = evaluate.load('bleu')
rouge = evaluate.load('rouge')
meteor = evaluate.load('meteor')

from evaluate import load
bertscore = load('bertscore')

bleurt = load('bleurt', module_type='metric')


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/gusloryst@GU.GU.SE/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Using default checkpoint 'bleurt-base-128' for sequence maximum length 128. You can use a bigger model for better results with e.g.: evaluate.load('bleurt', config_name='bleurt-large-512').


INFO:tensorflow:Reading checkpoint /home/gusloryst@GU.GU.SE/.cache/huggingface/metrics/bleurt/default/downloads/extracted/a276ad939c00eff0b3e8bc7cd147f7240d6d8e643a5ea5280abb4d3d73cdf75d/bleurt-base-128.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint bert_custom
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:bert_custom
INFO:tensorflow:... vocab_file:vocab.txt
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... do_lower_case:True
INFO:tensorflow:... max_seq_length:128
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating WordPiece tokenizer.
INFO:tensorflow:WordPiece tokenizer instantiated.
INFO:tensorflow:Creating Eager Mode predictor.
INFO:tensorflow:Loading model.


2026-05-14 17:33:03.249386: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


In [21]:
import json

In [23]:
from IPython import embed
from tqdm import tqdm
import torch.nn.functional as F
import os

def remove_relative_references(path):
    normalized_path = os.path.normpath(path)
    while normalized_path.startswith('..' + os.sep):
        normalized_path = normalized_path[3:]
    return normalized_path

output_results_loc = './out/'

for num, (dial_name, turns) in enumerate(meetup_turns.items()):

    #if num == 101:

    image_id = dial_name.split('/')[-1] + '-'
    for t in turns:
        
        turn = t['turn']
        name_to_save = image_id + str(turn) + '.json'
        name_to_save = remove_relative_references(name_to_save)

        visual_history = vis_context_path + image_id + str(turn) + '.jpg'
        textual_history = t['hist_msg']
        target_message = t['turn_message']

        # print(visual_history)
        # print(type(visual_history))
        visual_history = "./data/example.jpg"
        # print(visual_history)
        
        
        images_tensor = process_images(
            [Image.open(image) for image in [visual_history]],
            image_processor,
            model.config
        ).to(model.device, dtype=torch.float16)
        
        img = images_tensor


        #print('TARGET MESSAGE', target_message)
        chat_perplexity = []
        chat_generation = []
        if textual_history == []:
            chat_generation.append('The chat is empty.')
            chat_perplexity.append(target_message)
        else:
            for msg in textual_history:
                chat_perplexity.append(f'{msg}')
                chat_generation.append(f'{msg}')
            chat_perplexity.append(target_message)
        chat_generation = '\n'.join(chat_generation)
        chat_perplexity = '\n'.join(chat_perplexity)
        

        #display(Image.open(visual_history))
        

        prompt_for_perplexity = f"""
You are a helpful language and vision assistant. You see a chat between two people, A and B. They are playing a game in which they are trying to find each other in a house. What you see are the pictures of each room they have visited. The rooms visited by person A are shown in the top row, and the rooms visited by person B are shown in the bottom row. Pictures in each row are arranged in sequence from left to right, representing the order in which they were taken. Person A is currently in the room shown in the rightmost picture from the top row, and person B is currently in the room shown in the rightmost picture from the bottom row. A and B are having a chat and are trying to ensure that they are in the same room, i.e., they have to see the same picture. Each player does not see what the other player sees. Sometimes the chat is empty, which means that the players have not written any messages yet.

What do you think is the next message based on the information you have about the game, the players, the rooms they have visited, and their chat?

CHAT:

{chat_perplexity}
"""

        prompt_for_generation = f"""
You are a helpful language and vision assistant. You see a chat between two people, A and B. They are playing a game in which they are trying to find each other in a house. What you see are the pictures of each room they have visited. The rooms visited by person A are shown in the top row, and the rooms visited by person B are shown in the bottom row. Pictures in each row are arranged in sequence from left to right, representing the order in which they were taken. Person A is currently in the room shown in the rightmost picture from the top row, and person B is currently in the room shown in the rightmost picture from the bottom row. A and B are having a chat and are trying to ensure that they are in the same room, i.e., they have to see the same picture. Each player does not see what the other player sees. Sometimes the chat is empty, which means that the players have not written any messages yet.

What do you think is the next message based on the information you have about the game, the players, the rooms they have visited, and their chat?

CHAT:

{chat_generation}
"""

        full_input_ids = tokenizer_image_token(prompt_for_perplexity, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
        full_input_ids = full_input_ids.unsqueeze(0).to(model.device)
        #get the length of the target message
        targets_as_input_ids = tokenizer_image_token(target_message, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
        target_len = targets_as_input_ids.unsqueeze(0).to(model.device).shape[1]
        # calculate loss, i.e. for perplexity calculation
        # https://huggingface.co/docs/transformers/v4.37.2/en/perplexity
        # https://huggingface.co/spaces/evaluate-metric/perplexity
        targets_as_input_ids = tokenizer_image_token(target_message, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
        targets_as_input_ids = targets_as_input_ids.unsqueeze(0).to(model.device)

        initial_context_len = full_input_ids.shape[-1] - target_len
        seq_len = target_len
        nlls = []
        begin_loc = 0
        prev_end_loc = initial_context_len
        stride = 1
        for end_loc in range(initial_context_len, initial_context_len + seq_len, stride):
            input_ids = full_input_ids[:, begin_loc:end_loc].to(device)
            target_ids = input_ids.clone()
            target_ids[:, :-1] = -100
            with torch.no_grad():
                outputs = model(input_ids, images=img, image_sizes=[(336, 336)], labels=target_ids)
                loss = outputs.loss
                neg_log_likelihood = loss
            nlls.append(neg_log_likelihood)
            prev_end_loc = end_loc
            stride += 1
            if end_loc == initial_context_len:
                break
        ppl = torch.exp(torch.stack(nlls).mean())


        
        input_ids = tokenizer_image_token(prompt_for_generation, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
        input_ids = input_ids.unsqueeze(0).to(model.device)
        targets_as_input_ids = tokenizer_image_token(target_message, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
        targets_as_input_ids = targets_as_input_ids.unsqueeze(0).to(model.device)
        target_ids = targets_as_input_ids.clone()
        trg_len = len(targets_as_input_ids)
        #target_ids[:, :-trg_len] = -100

        generated_out = model.generate(inputs=input_ids,
                                       images=img,
                                       image_sizes=[(336, 336)],
                                       max_length=500,
                                       return_dict_in_generate=True,
                                       output_scores=True,
                                       num_beams=2)
        
        generated_response = tokenizer.batch_decode(generated_out.sequences, skip_special_tokens=True)[0]

        messages = re.split(r'(A:|B:)', generated_response)
        messages = [msg.strip() for msg in messages if msg.strip()]
        
        first_message = messages[0] + ' ' + messages[1].strip('"')

        results_dict = {}


        #print('-----')
        #print('perplexity', ppl)
        #print('model prediction', first_message)
        #print('gt', target_message)
        #print('-----')
        #print()

        results_bleu1 = bleu.compute(predictions=[first_message], references=[[target_message]], max_order=1)
        results_bleu2 = bleu.compute(predictions=[first_message], references=[[target_message]], max_order=2)
        results_bleu3 = bleu.compute(predictions=[first_message], references=[[target_message]], max_order=3)
        results_bleu4 = bleu.compute(predictions=[first_message], references=[[target_message]], max_order=4)
        results_rouge = rouge.compute(predictions=[first_message], references=[[target_message]])
        # results_meteor = meteor.compute(predictions=[first_message], references=[[target_message]])
        results_bertscore = bertscore.compute(predictions=[first_message], references=[[target_message]], lang="en")
        results_bleurt = bleurt.compute(predictions=[first_message], references=[target_message])

        results_dict['perplexity'] = ppl.item()
        results_dict['reference'] = target_message
        results_dict['hypothesis'] = first_message
        results_dict['bleu-1'] = results_bleu1['bleu']
        results_dict['bleu-2'] = results_bleu2['bleu']
        results_dict['bleu-3'] = results_bleu3['bleu']
        results_dict['bleu-4'] = results_bleu4['bleu']
        results_dict['rouge'] = results_rouge['rouge1']
        # results_dict['meteor'] = results_meteor['meteor']
        results_dict['bertscore'] = results_bertscore['f1']
        results_dict['bleurt'] = results_bleurt['scores']

        !
        with open(output_results_loc + name_to_save, 'w') as f:
            json.dump(results_dict, f)
        
    #break




KeyboardInterrupt: 

In [24]:
vis_context_path = './data/inputs/'

for dial_name, turns in meetup_turns.items():
    
    image_id = dial_name.split('/')[-1] + '-'
    for t in turns:
        
        turn = t['turn']

        visual_history = vis_context_path + image_id + str(turn) + '.jpg'
        textual_history = t['hist_msg']
        target_message = t['turn_message']

        prompt = f"""
        
        You are provided with the visual and textual history from the chat between two people, A and B.
        Each person has their own visual history, but they both share a textual history.
        The visual history consists of images arranged in a sequence from left to right, representing the order
        in which each person observed the images.
        Person A's images are displayed in the top row, while Person B's images are shown in the bottom row.
        The shared textual history comprises chat messages produced by A and B.
        If there is such a history, it is shown below.

        {textual_history}

        Your task is to predict the next message in this chat, based on the information you have about
        visual and textual history. Output only what you think is the next message in this chat.
        
        """

        args = type('Args', (), {
            "model_path": model_path,
            "model_base": None,
            "model_name": get_model_name_from_path(model_path),
            "query": prompt,
            "conv_mode": None,
            "image_file": Image.open(visual_history),
            "sep": ",",
            "temperature": 0,
            "top_p": None,
            "num_beams": 1,
            "max_new_tokens": 512
        })()
        
        out = eval_model(args)


        print(out)
        print()
    
    break

FileNotFoundError: [Errno 2] No such file or directory: '/home/gusloryst@GU.GU.SE/mgr/sim_nic_paper/data/inputs/2018-11-29 19-44-34-meetup 3.log-26.jpg'